# Spam/Ham Classifier — Training Notebook

This notebook extracts the dataset, preprocesses the text, trains a PyTorch feed-forward classifier over TF-IDF features, evaluates it, and runs a sample prediction.

## 1. Extract dataset

In [ ]:
import zipfile

zip_path = "/content/spam.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")

## 2. Imports & setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
import string

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

data = pd.read_csv('/content/spam.csv', encoding='latin-1')
data.head()

## 3. Preprocess text

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])  # remove punctuation
    tokens = word_tokenize(text)
    tokens_without_stop_words = [token for token in tokens if token not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens_without_stop_words]
    stemmed_tokens = [stemmer.stem(token) for token in lemmatized_tokens]
    return ' '.join(stemmed_tokens)

data['Cleaned'] = data['text'].apply(preprocess_text)

print(data['Cleaned'].head())

## 4. Feature extraction, dataset, and model

In [ ]:
# Feature extraction (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(data['Cleaned']).toarray()
y = data['label_num'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


class EmailDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def __len__(self):
        return len(self.X)


train_dataset = EmailDataset(X_train, y_train)
test_dataset = EmailDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


class SpamClassifier(nn.Module):
    def __init__(self, num_inputs):
        super().__init__()
        self.fc1 = nn.Linear(num_inputs, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc3(x))
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_inputs = X_train.shape[1]
model = SpamClassifier(num_inputs).to(device)

print(f"Using device: {device}")

## 5. Train

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
no_epochs = 10

for epoch in range(no_epochs):
    model.train()
    epoch_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels.unsqueeze(1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch [{epoch + 1}/{no_epochs}] - Loss: {epoch_loss / len(train_loader):.4f}")

## 6. Evaluate

In [ ]:
model.eval()  # Set the model to evaluation mode
correct = 0
total = 0
with torch.no_grad():  # Disable gradient calculation for evaluation
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (outputs > 0.5).float()  # Convert probabilities to binary predictions (0 or 1)

        total += labels.size(0)
        correct += (predicted == labels.unsqueeze(1)).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

## 7. Try a sample prediction

In [ ]:
test_email = 'Hi, you have the draw from lottomax and we will connect you soon if you further selected for the lottery, make your banking details ready'

def predict_email(email_text):
    model.eval()
    preprocessed_email = preprocess_text(email_text)
    input_vector = vectorizer.transform([preprocessed_email]).toarray()
    input_tensor = torch.tensor(input_vector, dtype=torch.float32).to(device)
    with torch.no_grad():
        output = model(input_tensor)
    predicted_label = 'Spam' if output.item() > 0.5 else 'Ham'
    return predicted_label

predicted_result = predict_email(test_email)

if predicted_result == 'Spam':
    print("The email is spam.")
else:
    print("The email is not spam.")

## 8. Save model + vectorizer (optional)

In [ ]:
import pickle

torch.save({"state_dict": model.state_dict(), "num_inputs": num_inputs}, "/content/spam_classifier.pkl")
with open("/content/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved model and vectorizer.")